# Preprocessing of Hourly Data

## Libraries

In [1]:
import pandas as pd
import os
from utils import read_files
from utils import float_to_int

## Traffic Data

### Read and Tidy Data

In [2]:
# this folder is not on GitHub, because it's too large
path = "data/traffic_h1/"

files_traffic = os.listdir(path)

for i in range(len(files_traffic)):
    files_traffic[i] = os.path.join(path, files_traffic[i])

df_traffic_raw = read_files(file_paths=files_traffic, drop_columns=["MSID", "MSName", "ZSID", "Achse", "HNr", "Hoehe", "EKoord", "NKoord", "Knummer", "Kname", "AnzDetektoren", "D1ID", "D2ID", "D3ID", "D4ID", "LieferDat", "AnzFahrzeugeStatus"], date_column="MessungDatZeit", UTC=False)

Create a map for the stations, that the stations of the traffic data can be merged with the stations of the pollutant and meteo data.

In [3]:
station_map = {
    "Rosengartenstrasse (Nordstrasse)": "Zch_Rosengartenstrasse",
    "Stampfenbachstrasse (Wasserwerkstrasse)": "Zch_Stampfenbachstrasse",
    "Schimmelstrasse (Werdstrasse)": "Zch_Schimmelstrasse"
    #"Zürichbergstrasse (Dreiwiesenstrasse)": "Zch_Heubeeribüel"
}

In [4]:
df_traffic_raw = df_traffic_raw[df_traffic_raw["ZSName"].isin(station_map)]

In [5]:
df_traffic_raw["Location"] = df_traffic_raw["ZSName"].map(station_map)

In [6]:
# select the necessary columns only
df_traffic_raw = df_traffic_raw.loc[:, ["Richtung", "Year", "Datetime", "Location", "AnzFahrzeuge"]]

In [7]:
df_traffic_raw.head()

,Richtung,Year,Datetime,Location,AnzFahrzeuge
614880,Bucheggplatz,2012,2011-12-31 23:00:00+00:00,Zch_Rosengartenstrasse,561.0
614881,Bucheggplatz,2012,2012-01-01 00:00:00+00:00,Zch_Rosengartenstrasse,1223.0
614882,Bucheggplatz,2012,2012-01-01 01:00:00+00:00,Zch_Rosengartenstrasse,989.0
614883,Bucheggplatz,2012,2012-01-01 02:00:00+00:00,Zch_Rosengartenstrasse,826.0
614884,Bucheggplatz,2012,2012-01-01 03:00:00+00:00,Zch_Rosengartenstrasse,823.0


### Check Datetime Column

Here we check whether the function "read_files" has handled the time zone conversions correctly.

In [8]:
# This code block checks in UTC. Maybe can delete that. In next block we check in local time zone.

df_traffic_raw["Date"] = pd.to_datetime(df_traffic_raw["Datetime"]).dt.date
df_traffic_raw["Hour"] = pd.to_datetime(df_traffic_raw["Datetime"]).dt.hour

hour_counts = (
    df_traffic_raw
    .groupby(["Location", "Date"])
    .size()
    .rename("n_hours")
    .reset_index()
)

missing_hours = hour_counts[hour_counts['n_hours'] != 48]
missing_hours

,Location,Date,n_hours
0,Zch_Rosengartenstrasse,2011-12-31,2
5114,Zch_Rosengartenstrasse,2025-12-31,46
5115,Zch_Schimmelstrasse,2011-12-31,2
10229,Zch_Schimmelstrasse,2025-12-31,46
10230,Zch_Stampfenbachstrasse,2011-12-31,2
15344,Zch_Stampfenbachstrasse,2025-12-31,46


As we converted the CET/CEST datetime column to UTC, there are two data points in 2011 and 46 only in 2025. This is correct. We check for 48 instead of 24 because for each day and hour, the traffic is measured in two directions. Therefore, each station and datetime consists of two data points. We will handle this later in the notebook.

In [9]:
# Check if there is all hours per each location-day combination, in local time zone.

# Make column with hour in tz Europe/Zurich
df_traffic_raw["Datetime_local"] = df_traffic_raw["Datetime"].dt.tz_convert("Europe/Zurich")
df_traffic_raw['Hour_local'] = df_traffic_raw['Datetime_local'].dt.hour
df_traffic_raw["Date_local"] = df_traffic_raw["Datetime_local"].dt.date

hour_counts = (
    df_traffic_raw
    .groupby(["Location", "Date_local"])
    .size()
    .rename("n_hours")
    .reset_index()
)

missing_hours = hour_counts[hour_counts['n_hours'] != 48]
missing_hours

,Location,Date_local,n_hours
84,Zch_Rosengartenstrasse,2012-03-25,46
301,Zch_Rosengartenstrasse,2012-10-28,50
455,Zch_Rosengartenstrasse,2013-03-31,46
665,Zch_Rosengartenstrasse,2013-10-27,50
819,Zch_Rosengartenstrasse,2014-03-30,46
...,...,...,...
14547,Zch_Stampfenbachstrasse,2023-10-29,50
14701,Zch_Stampfenbachstrasse,2024-03-31,46
14911,Zch_Stampfenbachstrasse,2024-10-27,50
15065,Zch_Stampfenbachstrasse,2025-03-30,46


Here we checked the data points in the local time zone. This returns 46 data points at the end of March and 50 at the end of October, which is correct because of the daylight saving time.

### Check Zero Values

Now we check the hourly traffic data for zero values.

In [10]:
print("df_traffic_raw shape:", df_traffic_raw.shape)

print(df_traffic_raw.isnull().sum())


df_traffic_raw shape: (736416, 10)
Richtung              0
Year                  0
Datetime              0
Location              0
AnzFahrzeuge      53826
Date                  0
Hour                  0
Datetime_local        0
Hour_local            0
Date_local            0
dtype: int64


In [11]:
(df_traffic_raw
.groupby(["Year", "Location"], as_index=False)
.agg(
        count_Richtung=("Richtung", "count"),
        count_Datetime=("Datetime", "count"),
        count_AnzFhz=("AnzFahrzeuge", "count"),
        sum_AnzFhz=("AnzFahrzeuge", "sum")))

,Year,Location,count_Richtung,count_Datetime,count_AnzFhz,sum_AnzFhz
0,2012,Zch_Rosengartenstrasse,17568,17568,14897,14584615.0
1,2012,Zch_Schimmelstrasse,17568,17568,17502,9764644.0
2,2012,Zch_Stampfenbachstrasse,17568,17568,17496,4952538.0
3,2013,Zch_Rosengartenstrasse,17520,17520,12397,12529021.0
4,2013,Zch_Schimmelstrasse,17520,17520,17504,9919183.0
5,2013,Zch_Stampfenbachstrasse,17520,17520,17504,4896925.0
6,2014,Zch_Rosengartenstrasse,17520,17520,0,0.0
7,2014,Zch_Schimmelstrasse,17520,17520,17506,9939737.0
8,2014,Zch_Stampfenbachstrasse,17520,17520,16914,4659701.0
9,2015,Zch_Rosengartenstrasse,17520,17520,5753,6212429.0


It seems like in the year 2014, the station at Rosengartenstrasse was not working at all. As the count of AnzFhz differs from the total count in most of the cases, there seem to be further missing data points.

### Check Number of Data Points

For each station and datetime, there are two values for number of vehicles, because the sensor measures in both directions. Therefore, we sum the number.

In [12]:
df_traffic = (df_traffic_raw
              .groupby(["Year", "Location", "Datetime"], as_index=False)
              .agg(n_vehicles=("AnzFahrzeuge", "sum")))

In [13]:
df_traffic.head()

,Year,Location,Datetime,n_vehicles
0,2012,Zch_Rosengartenstrasse,2011-12-31 23:00:00+00:00,1025.0
1,2012,Zch_Rosengartenstrasse,2012-01-01 00:00:00+00:00,2006.0
2,2012,Zch_Rosengartenstrasse,2012-01-01 01:00:00+00:00,1726.0
3,2012,Zch_Rosengartenstrasse,2012-01-01 02:00:00+00:00,1384.0
4,2012,Zch_Rosengartenstrasse,2012-01-01 03:00:00+00:00,1320.0


### Separate Datetime Column

For further analysis, separate the date time column.

In [14]:
df_traffic["Date"] = pd.to_datetime(df_traffic["Datetime"]).dt.date
df_traffic["Time"] = pd.to_datetime(df_traffic["Datetime"]).dt.time
df_traffic["Month"] = pd.to_datetime(df_traffic["Datetime"]).dt.month
df_traffic["Weekday"] = pd.to_datetime(df_traffic["Datetime"]).dt.weekday

In [15]:
df_traffic.head()

,Year,Location,Datetime,n_vehicles,Date,Time,Month,Weekday
0,2012,Zch_Rosengartenstrasse,2011-12-31 23:00:00+00:00,1025.0,2011-12-31,23:00:00,12,5
1,2012,Zch_Rosengartenstrasse,2012-01-01 00:00:00+00:00,2006.0,2012-01-01,00:00:00,1,6
2,2012,Zch_Rosengartenstrasse,2012-01-01 01:00:00+00:00,1726.0,2012-01-01,01:00:00,1,6
3,2012,Zch_Rosengartenstrasse,2012-01-01 02:00:00+00:00,1384.0,2012-01-01,02:00:00,1,6
4,2012,Zch_Rosengartenstrasse,2012-01-01 03:00:00+00:00,1320.0,2012-01-01,03:00:00,1,6


In [16]:
df_traffic.describe()

,Year,n_vehicles,Month,Weekday
count,368208.000000,368208.000000,368208.000000,368208.000000
mean,2018.499609,1121.548079,6.522487,2.999438
std,4.031522,944.483772,3.448829,2.000351
min,2012.000000,0.000000,1.000000,0.000000
25%,2015.000000,348.000000,4.000000,1.000000
50%,2018.500000,867.000000,7.000000,3.000000
75%,2022.000000,1642.000000,10.000000,5.000000
max,2025.000000,5331.000000,12.000000,6.000000


In [17]:
# check time during summer time
#df_traffic.loc[df_traffic['Datetime'].dt.date == pd.to_datetime('2024-06-18').date(), 'Datetime']

### Save Data Set

In [18]:
#df_traffic.to_csv("data/traffic_h1_2012_2024.csv", index=False)

## Air Pollution Data

### Read and Tidy Data

In [19]:
path = "data/air_pollutants_h1/"

files_pollutants = os.listdir(path)

for i in range(len(files_pollutants)):
    files_pollutants[i] = os.path.join(path, files_pollutants[i])

In [20]:
df_pollutants = read_files(file_paths=files_pollutants, drop_columns=["Status", "Intervall"], date_column="Datum")

df_pollutants.head()

,Datum,Location,Parameter,Einheit,Wert,Year,Datetime
0,2012-01-01T00:00+0100,Zch_Heubeeribüel,NO2,µg/m3,14.31,2012,2011-12-31 23:00:00+00:00
1,2012-01-01T00:00+0100,Zch_Heubeeribüel,NO,µg/m3,1.06,2012,2011-12-31 23:00:00+00:00
2,2012-01-01T00:00+0100,Zch_Heubeeribüel,NOx,ppb,8.33,2012,2011-12-31 23:00:00+00:00
3,2012-01-01T00:00+0100,Zch_Heubeeribüel,O3,µg/m3,36.52,2012,2011-12-31 23:00:00+00:00
4,2012-01-01T00:00+0100,Zch_Schimmelstrasse,NO2,µg/m3,39.96,2012,2011-12-31 23:00:00+00:00


In [21]:
param_units_pollutants = (
    df_pollutants[["Parameter", "Einheit"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

param_units_pollutants

,Parameter,Einheit
0,NO2,µg/m3
1,NO,µg/m3
2,NOx,ppb
3,O3,µg/m3
4,PM10,µg/m3
5,CO,mg/m3
6,SO2,µg/m3
7,PM2.5,µg/m3


### Create Pivot Table

In [22]:
df_pollutants_pivot = (
    df_pollutants
    .pivot(index=["Datetime", "Location", "Year"], columns="Parameter", values="Wert")
    .reset_index()
    .loc[:, ["Year", "Datetime", "Location", "NO2", "O3", "PM10"]]
)

df_pollutants_pivot.columns.name = None

df_pollutants_pivot["Location_Code"] = (
    df_pollutants_pivot["Location"].astype("category").cat.codes
)

df_pollutants_pivot.tail()

,Year,Datetime,Location,NO2,O3,PM10,Location_Code
475537,2025,2025-12-31 20:00:00+00:00,Zch_Stampfenbachstrasse,60.56,3.22,58.92,3
475538,2025,2025-12-31 21:00:00+00:00,Zch_Heubeeribüel,19.39,37.23,36.28,0
475539,2025,2025-12-31 21:00:00+00:00,Zch_Rosengartenstrasse,61.88,4.00,74.86,1
475540,2025,2025-12-31 21:00:00+00:00,Zch_Schimmelstrasse,61.53,2.29,75.61,2
475541,2025,2025-12-31 21:00:00+00:00,Zch_Stampfenbachstrasse,54.46,4.00,73.77,3


In [23]:
# TODO: check/ask why there are negative values
# TODO: discuss if start in 2014, not 2012 (bc of Rosengartenstrasse, see below)

df_pollutants_pivot.describe()

,Year,NO2,O3,PM10,Location_Code
count,475542.000000,468368.000000,469227.000000,362019.000000,475542.000000
mean,2018.696103,28.370813,47.245470,16.095715,1.516190
std,3.942031,20.044909,31.372212,11.622089,1.132308
min,2012.000000,-0.630000,-2.530000,-267.300000,0.000000
25%,2015.000000,12.110000,21.590000,8.680000,0.000000
50%,2019.000000,24.160000,46.390000,13.710000,2.000000
75%,2022.000000,40.480000,67.860000,20.550000,3.000000
max,2025.000000,387.210000,196.510000,593.440000,3.000000


### Check Zero Values and Number of Data Points

In [24]:
print("df_pollutants_pivot shape:", df_pollutants_pivot.shape)

print(df_pollutants_pivot.isnull().sum())

(df_pollutants_pivot
.groupby(["Year", "Location"], as_index=False)
.agg(count_Datetime=("Datetime", "count"),
     count_NO2=("NO2", "count"),
     count_O3=("O3", "count"),
     count_PM10=("PM10", "count")))

df_pollutants_pivot shape: (475542, 7)
Year                  0
Datetime              0
Location              0
NO2                7174
O3                 6315
PM10             113523
Location_Code         0
dtype: int64


,Year,Location,count_Datetime,count_NO2,count_O3,count_PM10
0,2012,Zch_Heubeeribüel,8784,8688,8641,0
1,2012,Zch_Schimmelstrasse,8784,8688,8698,8532
2,2012,Zch_Stampfenbachstrasse,8784,8739,8755,8482
3,2013,Zch_Heubeeribüel,8760,8655,8704,0
4,2013,Zch_Rosengartenstrasse,2146,2131,2132,2006
5,2013,Zch_Schimmelstrasse,8760,8661,8693,8641
6,2013,Zch_Stampfenbachstrasse,8760,8711,8736,8568
7,2014,Zch_Heubeeribüel,8760,8669,8704,0
8,2014,Zch_Rosengartenstrasse,8760,8687,8661,8387
9,2014,Zch_Schimmelstrasse,8760,8669,8703,8708


It seems like measuring PM10 at Heubeeribüel started in 2024 only. Furthermore, there are no measurements for Rosengartenstrasse in 2012 and only a few in 2013.

### Save Data Set

In [25]:
#df_pollutants_pivot.to_csv("data/pollution_h1_2012_2024.csv", index=False)

## Meteo Data

### Read and Tidy Data

In [26]:
path = "data/ugz_meteo_h1/"

files_pollutants = os.listdir(path)

for i in range(len(files_pollutants)):
    files_pollutants[i] = os.path.join(path, files_pollutants[i])

In [27]:
df_meteo = read_files(file_paths=files_pollutants, drop_columns=["Status", "Intervall"], date_column="Datum")

df_meteo.head()

,Datum,Location,Parameter,Einheit,Wert,Year,Datetime
0,2012-01-01T00:00+0100,Zch_Schimmelstrasse,Hr,%Hr,90.84,2012,2011-12-31 23:00:00+00:00
1,2012-01-01T00:00+0100,Zch_Schimmelstrasse,RainDur,min,6.27,2012,2011-12-31 23:00:00+00:00
2,2012-01-01T00:00+0100,Zch_Schimmelstrasse,T,°C,7.92,2012,2011-12-31 23:00:00+00:00
3,2012-01-01T00:00+0100,Zch_Schimmelstrasse,WD,°,39.67,2012,2011-12-31 23:00:00+00:00
4,2012-01-01T00:00+0100,Zch_Schimmelstrasse,WVv,m/s,0.35,2012,2011-12-31 23:00:00+00:00


In [28]:
param_units_meteo = (
    df_meteo[["Parameter", "Einheit"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

param_units_meteo

,Parameter,Einheit
0,Hr,%Hr
1,RainDur,min
2,T,°C
3,WD,°
4,WVv,m/s
5,p,hPa
6,WVs,m/s
7,StrGlo,W/m2


### Create Pivot Table

In [29]:
df_meteo_pivot = (
    df_meteo
    .pivot(index=["Datetime", "Location", "Year"], columns="Parameter", values="Wert")
    .reset_index()
    .loc[:, ["Year", "Datetime", "Location", "Hr", "RainDur", "T", "WVv", "StrGlo"]]
)

df_meteo_pivot.columns.name = None

df_meteo_pivot.tail()

,Year,Datetime,Location,Hr,RainDur,T,WVv,StrGlo
370345,2025,2025-12-31 20:00:00+00:00,Zch_Stampfenbachstrasse,62.60,0.0,-2.14,0.37,0.02
370346,2025,2025-12-31 21:00:00+00:00,Zch_Heubeeribüel,83.00,NaN,-3.67,NaN,NaN
370347,2025,2025-12-31 21:00:00+00:00,Zch_Rosengartenstrasse,66.20,0.0,-3.28,0.06,NaN
370348,2025,2025-12-31 21:00:00+00:00,Zch_Schimmelstrasse,62.25,0.0,-2.36,0.38,NaN
370349,2025,2025-12-31 21:00:00+00:00,Zch_Stampfenbachstrasse,64.58,0.0,-2.63,1.11,0.01


In [30]:
df_meteo_pivot.describe()

,Year,Hr,RainDur,T,WVv,StrGlo
count,370350.000000,364223.000000,349997.000000,366335.000000,333620.000000,122565.000000
mean,2019.036128,70.547796,5.108608,12.075421,0.867922,139.496141
std,4.005719,17.550612,14.383778,7.941476,0.915509,227.447524
min,2012.000000,12.700000,0.000000,-13.250000,0.000000,0.000000
25%,2016.000000,58.730000,0.000000,5.800000,0.250000,0.020000
50%,2019.000000,73.910000,0.000000,11.670000,0.550000,4.250000
75%,2023.000000,84.580000,0.000000,17.940000,1.170000,193.710000
max,2025.000000,103.560000,137.800000,36.550000,10.560000,1060.750000


### Check Zero Values and Number of Data Points

In [31]:
print("df_meteo_pivot shape:", df_pollutants_pivot.shape)

print(df_meteo_pivot.isnull().sum())

(df_meteo_pivot
.groupby(["Year", "Location"], as_index=False)
.agg(count_Datetime=("Datetime", "count"),
     count_Hr=("Hr", "count"),
     count_RainDur=("RainDur", "count"),
     count_T=("T", "count"),
     count_WVv=("WVv", "count"),
     count_StrGlo=("StrGlo", "count")))

df_meteo_pivot shape: (475542, 7)
Year             0
Datetime         0
Location         0
Hr            6127
RainDur      20353
T             4015
WVv          36730
StrGlo      247785
dtype: int64


,Year,Location,count_Datetime,count_Hr,count_RainDur,count_T,count_WVv,count_StrGlo
0,2012,Zch_Schimmelstrasse,8784,8757,8779,8757,8775,0
1,2012,Zch_Stampfenbachstrasse,8784,8775,8781,8775,8781,8781
2,2013,Zch_Rosengartenstrasse,2146,2004,2144,2004,0,0
3,2013,Zch_Schimmelstrasse,8760,8732,8756,8732,8756,0
4,2013,Zch_Stampfenbachstrasse,8760,8744,8757,8744,8750,8757
5,2014,Zch_Rosengartenstrasse,8760,8757,8759,8757,0,0
6,2014,Zch_Schimmelstrasse,8760,8738,8758,8738,8752,0
7,2014,Zch_Stampfenbachstrasse,8760,8657,8754,8703,8754,8754
8,2015,Zch_Rosengartenstrasse,8760,8680,8684,8680,5181,0
9,2015,Zch_Schimmelstrasse,8760,8583,8597,8583,8587,0


StrGlo is measured for Stampfenbachstrasse only.

### Save Data Set

In [32]:
#df_meteo_pivot.to_csv("data/meteo_h1_2012_2024.csv", index=False)

## Merge Data Sets

In [33]:
# Combine all datasets across days and locations
df_combined = df_pollutants_pivot.merge(df_meteo_pivot, on=["Datetime", "Location"], how="outer") \
    .merge(df_traffic, on=["Datetime", "Location"], how="outer")

# remove the duplicated column "Year"
df_combined = df_combined.drop(columns=["Year_x", "Year_y"])

# remove Location Heubeeribüel
df_combined = df_combined[df_combined["Location"] != "Zch_Heubeeribüel"]

In [34]:
df_combined = float_to_int(df_combined, ["Location_Code", "Year", "n_vehicles", "Month", "Weekday"])

df_combined.tail()

,Datetime,Location,NO2,O3,PM10,Location_Code,Hr,RainDur,T,WVv,StrGlo,Year,n_vehicles,Date,Time,Month,Weekday
490938,2025-12-31 21:00:00+00:00,Zch_Schimmelstrasse,61.53,2.29,75.61,2,62.25,0.0,-2.36,0.38,NaN,2025,945,2025-12-31,21:00:00,12,2
490939,2025-12-31 21:00:00+00:00,Zch_Stampfenbachstrasse,54.46,4.00,73.77,3,64.58,0.0,-2.63,1.11,0.01,2025,484,2025-12-31,21:00:00,12,2
490940,2025-12-31 22:00:00+00:00,Zch_Rosengartenstrasse,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,2025,1583,2025-12-31,22:00:00,12,2
490941,2025-12-31 22:00:00+00:00,Zch_Schimmelstrasse,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,2025,1036,2025-12-31,22:00:00,12,2
490942,2025-12-31 22:00:00+00:00,Zch_Stampfenbachstrasse,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,2025,706,2025-12-31,22:00:00,12,2


In [35]:
print("df_combined shape:", df_combined.shape)

print(df_combined.isnull().sum())

df_combined shape: (368208, 17)
Datetime              0
Location              0
NO2               20450
O3                20455
PM10              23702
Location_Code     15401
Hr                21521
RainDur           18211
T                 19409
WVv               34588
StrGlo           245643
Year                  0
n_vehicles            0
Date                  0
Time                  0
Month                 0
Weekday               0
dtype: int64


In [36]:
(df_combined
.groupby(["Year", "Location"], as_index=False)
.agg(
    count_Datetime=("Datetime", "count"),
     count_NO2=("NO2", "count"),
     count_O3=("O3", "count"),
     count_PM10=("PM10", "count"),
     count_Hr=("Hr", "count"),
     count_RainDur=("RainDur", "count"),
     count_T=("T", "count"),
     count_WVv=("WVv", "count"),
     count_StrGlo=("StrGlo", "count")))

,Year,Location,count_Datetime,count_NO2,count_O3,count_PM10,count_Hr,count_RainDur,count_T,count_WVv,count_StrGlo
0,2012,Zch_Rosengartenstrasse,8784,0,0,0,0,0,0,0,0
1,2012,Zch_Schimmelstrasse,8784,8688,8698,8532,8757,8779,8757,8775,0
2,2012,Zch_Stampfenbachstrasse,8784,8739,8755,8482,8775,8781,8775,8781,8781
3,2013,Zch_Rosengartenstrasse,8760,2131,2132,2006,2004,2144,2004,0,0
4,2013,Zch_Schimmelstrasse,8760,8661,8693,8641,8732,8756,8732,8756,0
5,2013,Zch_Stampfenbachstrasse,8760,8711,8736,8568,8744,8757,8744,8750,8757
6,2014,Zch_Rosengartenstrasse,8760,8687,8661,8387,8757,8759,8757,0,0
7,2014,Zch_Schimmelstrasse,8760,8669,8703,8708,8738,8758,8738,8752,0
8,2014,Zch_Stampfenbachstrasse,8760,8707,8678,8547,8657,8754,8703,8754,8754
9,2015,Zch_Rosengartenstrasse,8760,8581,8611,8606,8680,8684,8680,5181,0


### Save Data Set

In [37]:
#df_combined.to_csv("data/combined_h1_2012_2024.csv", index=False)